# CatBoost Hyperparameter Experiments

Our baseline `CatBoost` achieved an OOF ROC-AUC of 0.946094.

We will test whether shallower trees, a lower learning rate, and stronger L2 regularization improve generalization.

We will keep the same features and cross-validation splits to make the results comparable.

In [5]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pathlib import Path

SEED = 42
BASELINE_AUC = 0.946094

root = Path("../playground-series-s6e11")
train_df = pd.read_csv(root / "train.csv")

X = train_df.drop(columns=["id", "is_fraud"]).copy()
y = train_df["is_fraud"]

cat_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)

for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

experiments = [
    {"name": "depth3", "depth": 3, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"name": "depth3_l2_10", "depth": 3, "learning_rate": 0.05, "l2_leaf_reg": 10},
    {"name": "depth3_lr003_l2_10", "depth": 3, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"name": "depth4_lr003_l2_20", "depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 20},
]

In [6]:
results = []
oof_predictions = {}

for config in experiments:
    name = config["name"]
    print(f"\nExperiment: {name}")

    oof = np.zeros(len(X))
    fold_scores = []
    best_iterations = []

    params = {
        "iterations": 5000,
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": False,
        "thread_count": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
        model = CatBoostClassifier(**params)

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_features,
            eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
            early_stopping_rounds=150,
            use_best_model=True
        )

        pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = pred

        score = roc_auc_score(y.iloc[valid_idx], pred)

        fold_scores.append(score)
        best_iterations.append(model.best_iteration_)

        print(
            f"Fold {fold}: AUC={score:.6f}, "
            f"best_iteration={model.best_iteration_}"
        )

    oof_auc = roc_auc_score(y, oof)
    oof_predictions[name] = oof

    results.append({
        **config,
        "oof_auc": oof_auc,
        "delta": oof_auc - BASELINE_AUC,
        "avg_best_iteration": np.mean(best_iterations),
        "fold_std": np.std(fold_scores),
    })

    print(f"OOF AUC: {oof_auc:.6f} ({oof_auc - BASELINE_AUC:+.6f})")


Experiment: depth3
Fold 1: AUC=0.945096, best_iteration=3372
Fold 2: AUC=0.948038, best_iteration=2839
Fold 3: AUC=0.951339, best_iteration=2538
Fold 4: AUC=0.946796, best_iteration=2663
Fold 5: AUC=0.942617, best_iteration=3437
OOF AUC: 0.946730 (+0.000636)

Experiment: depth3_l2_10
Fold 1: AUC=0.945279, best_iteration=3452
Fold 2: AUC=0.948195, best_iteration=3123
Fold 3: AUC=0.951055, best_iteration=2218
Fold 4: AUC=0.946734, best_iteration=2651
Fold 5: AUC=0.942586, best_iteration=3445
OOF AUC: 0.946706 (+0.000612)

Experiment: depth3_lr003_l2_10
Fold 1: AUC=0.945054, best_iteration=4988
Fold 2: AUC=0.948209, best_iteration=4978
Fold 3: AUC=0.951274, best_iteration=4620
Fold 4: AUC=0.946651, best_iteration=3455
Fold 5: AUC=0.942491, best_iteration=4981
OOF AUC: 0.946693 (+0.000599)

Experiment: depth4_lr003_l2_20
Fold 1: AUC=0.944704, best_iteration=4410
Fold 2: AUC=0.948012, best_iteration=4965
Fold 3: AUC=0.950871, best_iteration=3479
Fold 4: AUC=0.946704, best_iteration=3295
Fo

In [7]:
results_df = pd.DataFrame(results)

display(
    results_df.sort_values("oof_auc", ascending=False)
    .round(6)
)

,name,depth,learning_rate,l2_leaf_reg,oof_auc,delta,avg_best_iteration,fold_std
0,depth3,3,0.05,3,0.946730,0.000636,2969.8,0.002917
1,depth3_l2_10,3,0.05,10,0.946706,0.000612,2977.8,0.002833
2,depth3_lr003_l2_10,3,0.03,10,0.946693,0.000599,4604.4,0.002954
3,depth4_lr003_l2_20,4,0.03,20,0.946482,0.000388,4225.0,0.002876


### Initial Findings

All tested configurations improved over the original `CatBoost` baseline of **0.946094**.

The best result came from `depth=3`, reaching **0.946730**. Increasing L2 regularization did not help further, and lowering the learning rate produced a similar result.

Shallower trees seem promising, so we will test depths 2 and 4 while keeping the other parameters unchanged.

In [35]:
from pathlib import Path

output_dir = Path("catboost_tuning_artifacts")
output_dir.mkdir(exist_ok=True)

pd.DataFrame({
    "id": train_df["id"],
    "is_fraud": y,
    **oof_predictions
}).to_csv(output_dir / "tuned_catboost_oof.csv", index=False)

results_df.to_csv(output_dir / "tuning_results.csv", index=False)

## Exploring Tree Depth

Our first experiments suggest that reducing tree depth helps `CatBoost` generalize better.

We will now compare depths 2 and 4 using the same learning rate and L2 regularization.

The depth-2 model achieved **0.946883**, improving over depth 3. Some folds reached the 5,000-iteration limit, so we will also test whether allowing more iterations helps.

We will include depth 1 to check whether even simpler trees perform better.

In [36]:
experiments = [
    {"name": "depth2", "depth": 2, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"name": "depth4", "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
]

In [37]:
results = []
oof_predictions = {}

for config in experiments:
    name = config["name"]
    print(f"\nExperiment: {name}")

    oof = np.zeros(len(X))
    fold_scores = []
    best_iterations = []

    params = {
        "iterations": 5000,
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": False,
        "thread_count": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
        model = CatBoostClassifier(**params)

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_features,
            eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
            early_stopping_rounds=150,
            use_best_model=True
        )

        pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = pred

        score = roc_auc_score(y.iloc[valid_idx], pred)

        fold_scores.append(score)
        best_iterations.append(model.best_iteration_)

        print(
            f"Fold {fold}: AUC={score:.6f}, "
            f"best_iteration={model.best_iteration_}"
        )

    oof_auc = roc_auc_score(y, oof)
    oof_predictions[name] = oof

    results.append({
        **config,
        "oof_auc": oof_auc,
        "delta": oof_auc - BASELINE_AUC,
        "avg_best_iteration": np.mean(best_iterations),
        "fold_std": np.std(fold_scores),
    })

    print(f"OOF AUC: {oof_auc:.6f} ({oof_auc - BASELINE_AUC:+.6f})")


Experiment: depth2
Fold 1: AUC=0.945384, best_iteration=4929
Fold 2: AUC=0.948444, best_iteration=4999
Fold 3: AUC=0.951294, best_iteration=3198
Fold 4: AUC=0.946910, best_iteration=3399
Fold 5: AUC=0.942658, best_iteration=4716
OOF AUC: 0.946883 (+0.000789)

Experiment: depth4
Fold 1: AUC=0.944928, best_iteration=2083
Fold 2: AUC=0.948280, best_iteration=2602
Fold 3: AUC=0.951169, best_iteration=1706
Fold 4: AUC=0.946606, best_iteration=1344
Fold 5: AUC=0.942592, best_iteration=2873
OOF AUC: 0.946658 (+0.000564)


In [ ]:
experiments = [
    {"name": "depth1_8000", "depth": 1},
    {"name": "depth2_8000", "depth": 2},
]

new_results = []
new_oof = {}

for config in experiments:
    print(f"\nExperiment: {config['name']}")

    params = {
        "iterations": 8000,
        "depth": config["depth"],
        "learning_rate": 0.05,
        "l2_leaf_reg": 3,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": False,
        "thread_count": -1,
    }

    oof = np.zeros(len(X))
    fold_scores = []
    best_iterations = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
        model = CatBoostClassifier(**params)

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_features,
            eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
            early_stopping_rounds=150,
            use_best_model=True,
        )

        pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = pred

        auc = roc_auc_score(y.iloc[valid_idx], pred)

        fold_scores.append(auc)
        best_iterations.append(model.best_iteration_)

        print(
            f"Fold {fold}: AUC={auc:.6f}, "
            f"best_iteration={model.best_iteration_}"
        )

    oof_auc = roc_auc_score(y, oof)
    new_oof[config["name"]] = oof

    new_results.append({
        "experiment": config["name"],
        "oof_auc": oof_auc,
        "delta_vs_best": oof_auc - 0.946883,
        "avg_best_iteration": np.mean(best_iterations),
        "fold_std": np.std(fold_scores),
    })

    print(f"OOF AUC: {oof_auc:.6f}")


results_df = pd.DataFrame(new_results)
display(results_df.sort_values("oof_auc", ascending=False))


Experiment: depth1_8000
Fold 1: AUC=0.944314, best_iteration=7999
Fold 2: AUC=0.947847, best_iteration=7997
Fold 3: AUC=0.951323, best_iteration=7998
Fold 4: AUC=0.946478, best_iteration=7944
Fold 5: AUC=0.941990, best_iteration=7996
OOF AUC: 0.946355

Experiment: depth2_8000
Fold 1: AUC=0.945461, best_iteration=5581
Fold 2: AUC=0.948539, best_iteration=5700
Fold 3: AUC=0.951294, best_iteration=3198
Fold 4: AUC=0.946910, best_iteration=3399
Fold 5: AUC=0.942658, best_iteration=4716
OOF AUC: 0.946918


,experiment,oof_auc,delta_vs_best,avg_best_iteration,fold_std
1,depth2_8000,0.946918,0.000035,4518.8,0.002899
0,depth1_8000,0.946355,-0.000528,7986.8,0.003167


### Final CatBoost Results

The best `CatBoost` configuration was `depth=2` with up to 8,000 iterations, reaching an OOF `ROC-AUC` of **0.946918**.

Increasing the iteration limit from 5,000 to 8,000 improved the score by only **0.000035**. Depth 1 performed worse, reaching **0.946355**.

These results suggest that depth 2 is a good choice for this dataset. We will stop tuning `CatBoost` here and check whether similar regularization helps LightGBM and XGBoost.

## LightGBM and XGBoost Tuning

Shallower trees improved CatBoost, so we will test a similar idea with LightGBM and XGBoost.

For LightGBM, we will reduce the number of leaves and tree depth. For XGBoost, we will test smaller trees and stronger regularization.

We will keep the same 5-fold CV and save OOF predictions for later ensemble experiments.

In [ ]:
lgb_results = []
lgb_tuned_oof = {}

for config in lgb_experiments:
    name = config["name"]

    model_params = {
        "n_estimators": 5000,
        "learning_rate": 0.05,
        "objective": "binary",
        "metric": "auc",
        "random_state": SEED,
        "verbosity": -1,
        **{k: v for k, v in config.items() if k != "name"},
    }

    oof = np.zeros(len(y))

    print(f"\nLightGBM: {name}")

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_lgb, y), 1):
        model = lgb.LGBMClassifier(**model_params)

        model.fit(
            X_lgb.iloc[train_idx],
            y.iloc[train_idx],
            eval_set=[(X_lgb.iloc[valid_idx], y.iloc[valid_idx])],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(150, verbose=False)],
        )

        pred = model.predict_proba(X_lgb.iloc[valid_idx])[:, 1]
        oof[valid_idx] = pred

        auc = roc_auc_score(y.iloc[valid_idx], pred)

        print(f"Fold {fold}: AUC={auc:.6f}, best_iter={model.best_iteration_}")

    score = roc_auc_score(y, oof)
    lgb_tuned_oof[name] = oof

    lgb_results.append({
        "experiment": name,
        "oof_auc": score,
        "delta": score - 0.945159,
    })

    print(f"OOF AUC: {score:.6f}")

display(pd.DataFrame(lgb_results).sort_values("oof_auc", ascending=False))


LightGBM: small


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1: AUC=0.945506, best_iter=1369


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2: AUC=0.949314, best_iter=1488


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


... (22 outputs left out)

Fold 4: AUC=0.948105, best_iter=2784


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5: AUC=0.943233, best_iter=2736
OOF AUC: 0.947895


,experiment,oof_auc,delta
2,very_small,0.947895,0.002736
1,regularized,0.947248,0.002089
0,small,0.947235,0.002076


In [ ]:
import xgboost as xgb

X_xgb = pd.get_dummies(
    X,
    columns=cat_cols,
    dtype=int
)

print(X_xgb.shape)

(699960, 54)


In [ ]:
xgb_experiments = [
    {
        "name": "depth2",
        "max_depth": 2,
        "min_child_weight": 5,
        "reg_lambda": 5,
    },
    {
        "name": "depth3_regularized",
        "max_depth": 3,
        "min_child_weight": 10,
        "reg_lambda": 10,
    },
    {
        "name": "depth2_strong",
        "max_depth": 2,
        "min_child_weight": 20,
        "reg_lambda": 15,
    },
]

xgb_results = []
xgb_tuned_oof = {}

for config in xgb_experiments:
    name = config["name"]

    model_params = {
        "n_estimators": 5000,
        "learning_rate": 0.05,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": SEED,
        "n_jobs": -1,
        "early_stopping_rounds": 150,
        **{k: v for k, v in config.items() if k != "name"},
    }

    oof = np.zeros(len(y))

    print(f"\nXGBoost: {name}")

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_xgb, y), start=1
    ):
        model = xgb.XGBClassifier(**model_params)

        model.fit(
            X_xgb.iloc[train_idx],
            y.iloc[train_idx],
            eval_set=[(
                X_xgb.iloc[valid_idx],
                y.iloc[valid_idx]
            )],
            verbose=False,
        )

        pred = model.predict_proba(
            X_xgb.iloc[valid_idx]
        )[:, 1]

        oof[valid_idx] = pred
        auc = roc_auc_score(y.iloc[valid_idx], pred)

        print(
            f"Fold {fold}: AUC={auc:.6f}, "
            f"best_iter={model.best_iteration}"
        )

    score = roc_auc_score(y, oof)
    xgb_tuned_oof[name] = oof

    xgb_results.append({
        "experiment": name,
        "oof_auc": score,
        "delta": score - 0.945122,
    })

    print(f"OOF AUC: {score:.6f}")


XGBoost: depth2
Fold 1: AUC=0.945882, best_iter=1589
Fold 2: AUC=0.949247, best_iter=2985
Fold 3: AUC=0.951859, best_iter=2107
Fold 4: AUC=0.947400, best_iter=2378
Fold 5: AUC=0.943119, best_iter=2580
OOF AUC: 0.947467

XGBoost: depth3_regularized
Fold 1: AUC=0.945594, best_iter=1561
Fold 2: AUC=0.948894, best_iter=1407
Fold 3: AUC=0.951759, best_iter=1536
Fold 4: AUC=0.946654, best_iter=877
Fold 5: AUC=0.942572, best_iter=1356
OOF AUC: 0.947053

XGBoost: depth2_strong
Fold 1: AUC=0.946133, best_iter=2636
Fold 2: AUC=0.949138, best_iter=2636
Fold 3: AUC=0.952115, best_iter=3476
Fold 4: AUC=0.947247, best_iter=2854
Fold 5: AUC=0.943016, best_iter=3081
OOF AUC: 0.947502


In [ ]:
xgb_results_df = pd.DataFrame(xgb_results)

display(
    xgb_results_df.sort_values(
        "oof_auc", ascending=False
    )
)

pd.DataFrame({
    "id": train_df["id"],
    "is_fraud": y,
    **{f"xgb_{name}": pred
       for name, pred in xgb_tuned_oof.items()},
}).to_csv(
    output_dir / "xgboost_tuned_oof.csv",
    index=False
)

,experiment,oof_auc,delta
2,depth2_strong,0.947502,0.002380
0,depth2,0.947467,0.002345
1,depth3_regularized,0.947053,0.001931


### Boosting Results

Both LightGBM and XGBoost improved substantially over their original baselines.

The strongest LightGBM configuration used only four leaves and a maximum depth of two. XGBoost also performed best with depth-two trees and stronger regularization.

LightGBM achieved the highest OOF ROC-AUC, but we will also test whether combining the three models provides an additional improvement.

In [ ]:
from scipy.stats import rankdata

cb_oof = new_oof["depth2_8000"]
lgb_oof = lgb_tuned_oof["very_small"]
xgb_oof = xgb_tuned_oof["depth2_strong"]

mean_oof = (cb_oof + lgb_oof + xgb_oof) / 3

rank_oof = (
    rankdata(cb_oof) +
    rankdata(lgb_oof) +
    rankdata(xgb_oof)
) / 3

ensemble_results = pd.DataFrame([
    {"model": "CatBoost", "auc": roc_auc_score(y, cb_oof)},
    {"model": "LightGBM", "auc": roc_auc_score(y, lgb_oof)},
    {"model": "XGBoost", "auc": roc_auc_score(y, xgb_oof)},
    {"model": "Mean ensemble", "auc": roc_auc_score(y, mean_oof)},
    {"model": "Rank ensemble", "auc": roc_auc_score(y, rank_oof)},
])

display(ensemble_results.sort_values("auc", ascending=False))

,model,auc
1,LightGBM,0.947895
4,Rank ensemble,0.947861
3,Mean ensemble,0.947813
2,XGBoost,0.947502
0,CatBoost,0.946918


In [ ]:
import lightgbm as lgb
import xgboost as xgb

from catboost import CatBoostClassifier
from pathlib import Path

test_df = pd.read_csv(root / "test.csv")
sample = pd.read_csv(root / "sample_submission.csv")

# CatBoost
X_test_cb = test_df[X.columns].copy()

for col in cat_features:
    X_test_cb[col] = (
        X_test_cb[col].fillna("__MISSING__").astype(str)
    )

for col in X.select_dtypes(include="bool").columns:
    X_test_cb[col] = X_test_cb[col].astype(int)

# LightGBM
cat_cols = cat_features

X_lgb = X.copy()
X_test_lgb = X_test_cb.copy()

for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype("category")
    X_test_lgb[col] = pd.Categorical(
        X_test_lgb[col],
        categories=X_lgb[col].cat.categories
    )

# XGBoost
X_xgb = pd.get_dummies(X, columns=cat_cols, dtype=int)

X_test_xgb = pd.get_dummies(
    X_test_cb,
    columns=cat_cols,
    dtype=int
).reindex(columns=X_xgb.columns, fill_value=0)

print("Train:", X.shape)
print("Test:", X_test_cb.shape)

Train: (699960, 24)
Test: (299984, 24)


In [ ]:
cb_params = {
    "iterations": 8000,
    "depth": 2,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "verbose": False,
    "thread_count": -1,
}

lgb_params = {
    "n_estimators": 5000,
    "learning_rate": 0.05,
    "objective": "binary",
    "metric": "auc",
    "random_state": SEED,
    "verbosity": -1,
    "num_leaves": 4,
    "max_depth": 2,
    "min_child_samples": 100,
    "reg_lambda": 10,
}

xgb_params = {
    "n_estimators": 5000,
    "learning_rate": 0.05,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
    "early_stopping_rounds": 150,
    "max_depth": 2,
    "min_child_weight": 20,
    "reg_lambda": 15,
}

cb_test = np.zeros(len(test_df))
lgb_test = np.zeros(len(test_df))
xgb_test = np.zeros(len(test_df))

n_folds = cv.get_n_splits()

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), 1):
    print(f"\nFold {fold}")

    # CatBoost
    cb = CatBoostClassifier(**cb_params)

    cb.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
        cat_features=cat_features,
        eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
        early_stopping_rounds=150,
        use_best_model=True,
    )

    cb_test += cb.predict_proba(X_test_cb)[:, 1] / n_folds
    print("CatBoost done")

    # LightGBM
    lgb_model = lgb.LGBMClassifier(**lgb_params)

    lgb_model.fit(
        X_lgb.iloc[train_idx],
        y.iloc[train_idx],
        eval_set=[(
            X_lgb.iloc[valid_idx],
            y.iloc[valid_idx]
        )],
        eval_metric="auc",
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )

    lgb_test += (
        lgb_model.predict_proba(X_test_lgb)[:, 1] / n_folds
    )
    print("LightGBM done")

    # XGBoost
    xgb_model = xgb.XGBClassifier(**xgb_params)

    xgb_model.fit(
        X_xgb.iloc[train_idx],
        y.iloc[train_idx],
        eval_set=[(
            X_xgb.iloc[valid_idx],
            y.iloc[valid_idx]
        )],
        verbose=False,
    )

    xgb_test += (
        xgb_model.predict_proba(X_test_xgb)[:, 1] / n_folds
    )
    print("XGBoost done")


Fold 1
CatBoost done


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM done
XGBoost done

Fold 2
CatBoost done


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM done
XGBoost done

Fold 3
CatBoost done


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM done
XGBoost done

Fold 4
CatBoost done


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM done
XGBoost done

Fold 5
CatBoost done


/Users/ulad/kaggle/fraud_detection/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM done
XGBoost done


## Kaggle Submission

The three-model rank ensemble achieved an OOF ROC-AUC of **0.947861**, slightly below LightGBM alone at **0.947895**.

We will still submit the individual models and the ensemble to compare their performance on the public leaderboard.

For test predictions, we will average predictions from five models trained on the corresponding CV folds.

In [ ]:
mean_test = (cb_test + lgb_test + xgb_test) / 3

rank_test = (
    rankdata(cb_test) +
    rankdata(lgb_test) +
    rankdata(xgb_test)
) / (3 * len(test_df))

predictions = {
    "catboost": cb_test,
    "lightgbm": lgb_test,
    "xgboost": xgb_test,
    "ensemble_mean": mean_test,
    "ensemble_rank": rank_test,
}

submission_dir = Path("submissions")
submission_dir.mkdir(exist_ok=True)

for name, pred in predictions.items():
    submission = sample.copy()
    submission["is_fraud"] = pred

    path = submission_dir / f"submission_{name}.csv"
    submission.to_csv(path, index=False)

    print(f"Saved: {path}")

Saved: submissions/submission_catboost.csv
Saved: submissions/submission_lightgbm.csv
Saved: submissions/submission_xgboost.csv
Saved: submissions/submission_ensemble_mean.csv
Saved: submissions/submission_ensemble_rank.csv


## Experiment - Alternative Ensemble Combinations

Our tuned `XGBoost`, `LightGBM`, and `CatBoost` models achieved similar results on the public leaderboard, with `XGBoost` performing slightly better.

We will test whether removing `CatBoost` or assigning more weight to XGBoost improves OOF `ROC-AUC`.

We will compare both probability averaging and rank averaging.

In [30]:
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

cb = new_oof["depth2_8000"]
lgb_pred = lgb_tuned_oof["very_small"]
xgb_pred = xgb_tuned_oof["depth2_strong"]

def rank(pred):
    return rankdata(pred) / len(pred)

blends = {
    "XGB + LGB": (
        (xgb_pred + lgb_pred) / 2
    ),
    "XGB + CB": (
        (xgb_pred + cb) / 2
    ),
    "Weighted 50/30/20": (
        0.5 * xgb_pred +
        0.3 * lgb_pred +
        0.2 * cb
    ),
    "XGB + LGB (rank)": (
        rank(xgb_pred) + rank(lgb_pred)
    ) / 2,
    "XGB + CB (rank)": (
        rank(xgb_pred) + rank(cb)
    ) / 2,
    "Weighted 50/30/20 (rank)": (
        0.5 * rank(xgb_pred) +
        0.3 * rank(lgb_pred) +
        0.2 * rank(cb)
    ),
    "Original 3-model rank": (
        rank(xgb_pred) + rank(lgb_pred) + rank(cb)
    ) / 3,
}

blend_results = pd.DataFrame([
    {"blend": name, "oof_auc": roc_auc_score(y, pred)}
    for name, pred in blends.items()
])

display(blend_results.sort_values("oof_auc", ascending=False))

,blend,oof_auc
3,XGB + LGB (rank),0.947873
6,Original 3-model rank,0.947861
5,Weighted 50/30/20 (rank),0.947855
0,XGB + LGB,0.947825
2,Weighted 50/30/20,0.947813
4,XGB + CB (rank),0.947573
1,XGB + CB,0.947570


In [31]:
single_results = pd.DataFrame([
    {"model": "XGBoost", "oof_auc": roc_auc_score(y, xgb_pred)},
    {"model": "LightGBM", "oof_auc": roc_auc_score(y, lgb_pred)},
    {"model": "CatBoost", "oof_auc": roc_auc_score(y, cb)},
])

display(single_results.sort_values("oof_auc", ascending=False))

,model,oof_auc
1,LightGBM,0.947895
0,XGBoost,0.947502
2,CatBoost,0.946918


In [32]:
test_blends = {
    "xgb_lgb": (xgb_test + lgb_test) / 2,
    "xgb_cb": (xgb_test + cb_test) / 2,
    "weighted_50_30_20": (
        0.5 * xgb_test +
        0.3 * lgb_test +
        0.2 * cb_test
    ),
    "xgb_lgb_rank": (
        rank(xgb_test) + rank(lgb_test)
    ) / 2,
    "xgb_cb_rank": (
        rank(xgb_test) + rank(cb_test)
    ) / 2,
    "weighted_50_30_20_rank": (
        0.5 * rank(xgb_test) +
        0.3 * rank(lgb_test) +
        0.2 * rank(cb_test)
    ),
}

for name, pred in test_blends.items():
    submission = sample.copy()
    submission["is_fraud"] = pred

    path = submission_dir / f"submission_{name}.csv"
    submission.to_csv(path, index=False)

    print(f"Saved: {path}")

Saved: submissions/submission_xgb_lgb.csv
Saved: submissions/submission_xgb_cb.csv
Saved: submissions/submission_weighted_50_30_20.csv
Saved: submissions/submission_xgb_lgb_rank.csv
Saved: submissions/submission_xgb_cb_rank.csv
Saved: submissions/submission_weighted_50_30_20_rank.csv


## Final Findings

Hyperparameter tuning gave us the largest improvements in this project so far.

Reducing tree complexity improved all three boosting models. LightGBM achieved the best individual OOF ROC-AUC of **0.947895**, while XGBoost performed slightly better on the public leaderboard.

We also tested several ensemble combinations. Removing CatBoost from the rank ensemble gave a small improvement:

| Submission | Public ROC-AUC |
|---|---:|
| Original rank ensemble | 0.94699 |
| Tuned CatBoost | 0.94741 |
| Tuned LightGBM | 0.94810 |
| Tuned XGBoost | 0.94822 |
| Three-model rank ensemble | 0.94823 |
| **XGBoost + LightGBM rank ensemble** | **0.94827** |

The final improvement over our original submission was **0.00128 ROC-AUC**.

The difference between the two best ensembles is very small, so we cannot confidently say that one will generalize better to the private test set.

**Conclusion:** Simpler, better-regularized boosting models worked better than our original configurations. We will keep the XGBoost + LightGBM rank ensemble as our current best submission and stop tuning for now.